# Project 2: Healthcare Data Analysis
Analyze patient satisfaction, treatment costs, and clinical metrics to improve operational efficiency and patient outcomes.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_curve, auc

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("datasets/healthcare_data.csv")
df["AdmissionDate"] = pd.to_datetime(df["AdmissionDate"])
df["DischargeDate"] = pd.to_datetime(df["DischargeDate"])
print("Dataset Loaded. Shape:", df.shape)
df.head()

Dataset Loaded. Shape: (12000, 12)


,PatientID,Age,Gender,Department,AdmissionDate,DischargeDate,LengthOfStay,TreatmentCost,SatisfactionScore,StaffToPatientRatio,TelemedicineFollowUp,Readmitted
0,PAT-00001,18,Male,Pediatrics,2025-10-21,2025-10-26,5,29299.61,4.8,4.0,True,0
1,PAT-00002,14,Female,Pediatrics,2025-08-31,2025-09-02,2,42874.56,5.0,4.0,False,0
2,PAT-00003,65,Male,Cardiology,2025-01-19,2025-01-22,3,147298.66,3.7,6.0,True,0
3,PAT-00004,1,Female,Pediatrics,2025-03-05,2025-03-08,3,31771.25,5.0,4.0,True,0
4,PAT-00005,73,Male,Neurology,2025-03-02,2025-03-09,7,155156.14,3.8,6.0,True,0


## 1. Demographic & Clinical Department Analysis
We check satisfaction scores by department. We expect Pediatrics to have the highest satisfaction (average around 4.7/5.0).

In [2]:
# Satisfaction by Department
dept_sat = df.groupby("Department")["SatisfactionScore"].agg(["mean", "count", "std"]).reset_index()
print(dept_sat)

# Bar plot of satisfaction
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='Department', y='SatisfactionScore', hue='Department', palette='coolwarm', errorbar=None, legend=False)
plt.title("Average Patient Satisfaction by Department")
plt.ylim(1, 5)
plt.ylabel("Satisfaction Score (1-5)")
plt.tight_layout()
plt.savefig("visualizations/satisfaction_by_dept.png")
plt.savefig("../visualizations/satisfaction_by_dept.png")
plt.show()

         Department      mean  count       std
0        Cardiology  3.998448   2384  0.584240
1  General Medicine  3.978244   2381  0.598062
2         Neurology  3.993175   2403  0.573814
3          Oncology  4.001290   2404  0.560359
4        Pediatrics  4.661820   2428  0.295294


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_29436\2892420018.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Treatment Cost Analysis
Test the claim that weekend admissions are 30% more costly due to staffing and resources constraint.

In [3]:
# Add weekday/weekend flag
df['AdmissionDay'] = df['AdmissionDate'].dt.strftime('%A')
df['IsWeekendAdmission'] = df['AdmissionDay'].isin(['Saturday', 'Sunday'])

avg_cost = df.groupby('IsWeekendAdmission')['TreatmentCost'].mean().reset_index()
print("Average Treatment Cost - Weekday vs Weekend Admission:")
print(avg_cost)

# Hypotheses test: T-Test
weekday_cost = df[~df['IsWeekendAdmission']]['TreatmentCost']
weekend_cost = df[df['IsWeekendAdmission']]['TreatmentCost']

t_stat, p_val = stats.ttest_ind(weekend_cost, weekday_cost)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.4e}")
print(f"Cost Uplift on Weekends: {(weekend_cost.mean() - weekday_cost.mean())/weekday_cost.mean()*100:.2f}%")

# Plot cost comparison
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='IsWeekendAdmission', y='TreatmentCost', hue='IsWeekendAdmission', palette='Set2', legend=False)
plt.title("Treatment Cost Distribution by Admission Day")
plt.xticks([0, 1], ["Weekday", "Weekend"])
plt.ylabel("Treatment Cost (₹)")
plt.tight_layout()
plt.savefig("visualizations/weekend_cost_comparison.png")
plt.savefig("../visualizations/weekend_cost_comparison.png")
plt.show()

Average Treatment Cost - Weekday vs Weekend Admission:
   IsWeekendAdmission  TreatmentCost
0               False  101777.083045
1                True  131228.828559
T-statistic: 20.7225, P-value: 9.4243e-94
Cost Uplift on Weekends: 28.94%


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_29436\3022918504.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Readmission Factors
We analyze readmissions. The overall readmission rate is 8.3%.
We look at two major interventions: Telemedicine follow-up and early discharge (stay length <= 3 days).

In [4]:
# Readmission statistics
overall_readmit = df['Readmitted'].mean()
print(f"Overall Readmission Rate: {overall_readmit*100:.2f}%")

# Telemedicine follow-up impact
tele_readmit = df.groupby('TelemedicineFollowUp')['Readmitted'].mean().reset_index()
print("\nReadmission Rate by Telemedicine Follow-Up:")
print(tele_readmit)

# Length of Stay impact
early_discharge = df.groupby(df['LengthOfStay'] <= 3)['Readmitted'].mean().reset_index()
print("\nReadmission Rate by Early Discharge (<= 3 days):")
print(early_discharge)

# Visualizing Readmission rates by telemedicine follow-up
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='TelemedicineFollowUp', y='Readmitted', hue='LengthOfStay', errorbar=None, palette='pastel')
plt.title("Readmission Rates by Telemedicine & Length of Stay")
plt.ylabel("Readmission Rate")
plt.tight_layout()
plt.savefig("visualizations/readmission_factors.png")
plt.savefig("../visualizations/readmission_factors.png")
plt.show()

Overall Readmission Rate: 8.93%

Readmission Rate by Telemedicine Follow-Up:
   TelemedicineFollowUp  Readmitted
0                 False    0.108539
1                  True    0.060384

Readmission Rate by Early Discharge (<= 3 days):
   LengthOfStay  Readmitted
0         False    0.097661
1          True    0.076162


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_29436\1510658519.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Machine Learning Model: Predicting Readmission Risk
We train a classifier to predict whether a patient is at risk of readmission.

In [5]:
# Preprocessing for ML
X = df[['Age', 'LengthOfStay', 'TreatmentCost', 'StaffToPatientRatio', 'TelemedicineFollowUp']]
# Encode boolean telemedicine
X = pd.get_dummies(X, columns=['TelemedicineFollowUp'], drop_first=True, dtype=int)
y = df['Readmitted']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

# Plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("visualizations/roc_curve.png")
plt.savefig("../visualizations/roc_curve.png")
plt.show()

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.99      0.95      3278
           1       0.10      0.01      0.01       322

    accuracy                           0.91      3600
   macro avg       0.50      0.50      0.48      3600
weighted avg       0.84      0.91      0.87      3600



C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_29436\1554860308.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
